In [1]:
%pip install -q delta-spark pyspark

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 1.1 MB/s eta 0:00:00


Restart the kernel after installation, then run the remaining cells.

In [5]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip
builder=(SparkSession.builder.appName("Week7 Delta Merge")
.config("spark.sql.extensions","io.delta.sql.DeltaSparkSessionExtension")
.config("spark.sql.catalog.spark_catalog","org.apache.spark.sql.delta.catalog.DeltaCatalog"))
spark=configure_spark_with_delta_pip(builder).getOrCreate()
print(spark)

In [8]:
master = spark.read.csv(
    "/content/customer_master.csv",
    header=True,
    inferSchema=True
)

master.show(5)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset 

In [10]:
from pyspark.sql.functions import col

master = master.dropDuplicates().fillna({
    "Profit": 0,
    "Discount": 0
})

# Rename columns by replacing spaces with underscores
for c in master.columns:
    master = master.withColumnRenamed(c, c.replace(" ", "_"))

master.printSchema()

root
 |-- Row_ID: integer (nullable = true)
 |-- Order_ID: string (nullable = true)
 |-- Order_Date: string (nullable = true)
 |-- Ship_Date: string (nullable = true)
 |-- Ship_Mode: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal_Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product_ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product_Name: string (nullable = true)
 |-- Sales: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Discount: string (nullable = false)
 |-- Profit: double (nullable = false)



In [11]:
master.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/content/delta_customer")

In [12]:
incremental = spark.read.csv(
    "/content/customer_incremental.csv",
    header=True,
    inferSchema=True
)

for c in incremental.columns:
    incremental = incremental.withColumnRenamed(c, c.replace(" ", "_"))

incremental.show()

+------+--------------+----------+----------+--------------+-----------+-------------+-----------+-------------+---------+-------------+-----------+-------+---------------+---------------+------------+----------------+-----+--------+--------+------+
|Row_ID|      Order_ID|Order_Date| Ship_Date|     Ship_Mode|Customer_ID|Customer_Name|    Segment|      Country|     City|        State|Postal_Code| Region|     Product_ID|       Category|Sub-Category|    Product_Name|Sales|Quantity|Discount|Profit|
+------+--------------+----------+----------+--------------+-----------+-------------+-----------+-------------+---------+-------------+-----------+-------+---------------+---------------+------------+----------------+-----+--------+--------+------+
|     1|CA-2016-152156|08-11-2016|11-11-2016|  Second Class|   CG-12520|  Claire Gute|   Consumer|United States|Henderson|     Kentucky|      42420|  South|FUR-BO-10001798|      Furniture|   Bookcases|Bookcase Updated|  300|       2|       0|    50|


In [13]:
from delta.tables import DeltaTable

deltaTable = DeltaTable.forPath(spark, "/content/delta_customer")

(
    deltaTable.alias("target")
    .merge(
        incremental.alias("source"),
        "target.Row_ID = source.Row_ID"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [14]:
final_df = spark.read.format("delta").load("/content/delta_customer")

final_df.show()

print("Rows:", final_df.count())
print("Unique Row IDs:", final_df.dropDuplicates(["Row_ID"]).count())

+------+--------------+----------+----------+--------------+-----------+----------------+-----------+-------------+-------------+-------------+-----------+-------+---------------+---------------+------------+--------------------+-----------------+--------+--------+---------+
|Row_ID|      Order_ID|Order_Date| Ship_Date|     Ship_Mode|Customer_ID|   Customer_Name|    Segment|      Country|         City|        State|Postal_Code| Region|     Product_ID|       Category|Sub-Category|        Product_Name|            Sales|Quantity|Discount|   Profit|
+------+--------------+----------+----------+--------------+-----------+----------------+-----------+-------------+-------------+-------------+-----------+-------+---------------+---------------+------------+--------------------+-----------------+--------+--------+---------+
|    14|CA-2016-161389| 12/5/2016|12/10/2016|Standard Class|   IM-15070|    Irene Maddox|   Consumer|United States|      Seattle|   Washington|      98103|   West|OFF-BI-10